# LeetCode 1343 — Number of Sub-arrays of Size K and Average Greater than or Equal to Threshold bằng Python

Link đề: https://leetcode.com/problems/number-of-sub-arrays-of-size-k-and-average-greater-than-or-equal-to-threshold/ · Độ khó: **Medium** · Chủ đề: Array, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def numOfSubarrays(self, arr: list[int], k: int, threshold: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Biến điều kiện trung bình `sum/k >= threshold` thành điều kiện **số nguyên** `sum >= k·threshold` (tránh số thực, tránh phép chia).
2. Đi từ **brute force** `O(n·k)` → **tổng tiền tố (prefix sum)** `O(n)` → **cửa sổ trượt kích thước cố định** `O(n)`, `O(1)` bộ nhớ.
3. Nắm bất biến của cửa sổ trượt: "thêm phần tử mới bên phải, bỏ phần tử cũ bên trái" và đếm đúng **`n − k + 1`** cửa sổ.
4. Biết kỹ thuật **trừ ngưỡng** (`arr[i] − threshold`) để đưa bài về "tổng ≥ 0" — mở đường cho các biến thể độ dài bất kỳ.
5. Tránh lỗi: dùng `>`, bỏ sót cửa sổ cuối, bỏ nhầm phần tử khi trượt, tính lại `sum(arr[i:i+k])` mỗi bước.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `arr` và hai số nguyên `k`, `threshold`. Đếm số **mảng con liên tiếp có độ dài đúng `k`** mà **trung bình cộng** `≥ threshold`.

**Ví dụ 1:** `arr = [2,2,2,2,5,5,5,8], k = 3, threshold = 4` → `3`
(Các mảng con `[2,5,5]`, `[5,5,5]`, `[5,5,8]` có trung bình 4, 5, 6; các mảng con độ dài 3 khác có trung bình < 4.)

**Ví dụ 2:** `arr = [11,13,17,23,29,31,7,5,2,3], k = 3, threshold = 5` → `6`
(6 mảng con đầu tiên có trung bình > 5; lưu ý trung bình không nhất thiết là số nguyên.)

**Ràng buộc:** `1 <= arr.length <= 10^5`; `1 <= arr[i] <= 10^4`; `1 <= k <= arr.length`; `0 <= threshold <= 10^4`.

In [1]:
import bisect
import random
import time
from itertools import accumulate
from typing import List

examples = [
    ([2, 2, 2, 2, 5, 5, 5, 8], 3, 4, 3),
    ([11, 13, 17, 23, 29, 31, 7, 5, 2, 3], 3, 5, 6),
]
for arr, k, t, want in examples:
    print(f"arr={arr}, k={k}, threshold={t} -> mong đợi {want} | số cửa sổ = n-k+1 = {len(arr) - k + 1}")

arr=[2, 2, 2, 2, 5, 5, 5, 8], k=3, threshold=4 -> mong đợi 3 | số cửa sổ = n-k+1 = 6
arr=[11, 13, 17, 23, 29, 31, 7, 5, 2, 3], k=3, threshold=5 -> mong đợi 6 | số cửa sổ = n-k+1 = 8


## 3. Kiến thức nền

### 3.1 Mảng con liên tiếp độ dài `k` và cắt lát

Mảng con bắt đầu tại `i` là `arr[i:i+k]` (chỉ số `i..i+k−1`). `i` chạy từ `0` đến `n − k` ⇒ có **`n − k + 1`** mảng con.

In [2]:
arr = [2, 2, 2, 2, 5, 5, 5, 8]
k = 3
for i in range(len(arr) - k + 1):
    print(i, arr[i:i + k])

0 [2, 2, 2]
1 [2, 2, 2]
2 [2, 2, 5]
3 [2, 5, 5]
4 [5, 5, 5]
5 [5, 5, 8]


### 3.2 Đổi điều kiện trung bình sang số nguyên

`sum / k >= threshold` ⇔ `sum >= k * threshold` (vì `k > 0`). Dạng nhân **không có phép chia**, toàn số nguyên ⇒ không lo sai số dấu phẩy động, và nhanh hơn.

In [3]:
s, k, t = 12, 3, 4
print(s / k >= t, s >= k * t)
s = 14
print(f"trung bình = {s / k:.4f}", s / k >= 5, s >= k * 5)

True True
trung bình = 4.6667 False False


### 3.3 Tổng tiền tố (prefix sum)

`P[0] = 0`, `P[i+1] = P[i] + arr[i]` ⇒ tổng đoạn `arr[l..r]` = `P[r+1] − P[l]` trong `O(1)`. `itertools.accumulate(arr, initial=0)` tạo `P` nhanh.

In [4]:
P = list(accumulate(arr, initial=0))
print("P =", P)
l, r = 4, 6
print(f"tổng arr[{l}..{r}] = P[{r + 1}] - P[{l}] = {P[r + 1] - P[l]} | kiểm tra: {sum(arr[l:r + 1])}")

P = [0, 2, 4, 6, 8, 13, 18, 23, 31]
tổng arr[4..6] = P[7] - P[4] = 15 | kiểm tra: 15


### 3.4 Cửa sổ trượt kích thước cố định

Giữ biến `window` = tổng `k` phần tử hiện tại. Trượt sang phải một bước: **cộng** phần tử mới vào, **trừ** phần tử vừa rơi ra bên trái. Mỗi bước `O(1)`.

In [5]:
window = sum(arr[:k])
print("cửa sổ 0:", arr[:k], "tổng", window)
for i in range(k, len(arr)):
    window += arr[i] - arr[i - k]
    print(f"cửa sổ {i - k + 1}: thêm {arr[i]}, bỏ {arr[i - k]} -> {arr[i - k + 1:i + 1]} tổng {window}")

cửa sổ 0: [2, 2, 2] tổng 6
cửa sổ 1: thêm 2, bỏ 2 -> [2, 2, 2] tổng 6
cửa sổ 2: thêm 5, bỏ 2 -> [2, 2, 5] tổng 9
cửa sổ 3: thêm 5, bỏ 2 -> [2, 5, 5] tổng 12
cửa sổ 4: thêm 5, bỏ 2 -> [5, 5, 5] tổng 15
cửa sổ 5: thêm 8, bỏ 5 -> [5, 5, 8] tổng 18


### 3.5 `sum` trên generator và đếm bằng `sum(điều_kiện)`

Trong Python, `True == 1`, `False == 0` ⇒ `sum(cond for ...)` đếm số lần điều kiện đúng.

In [6]:
print(sum(x >= 5 for x in arr), [x >= 5 for x in arr])

4 [False, False, False, False, True, True, True, True]


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1**: `arr = [2,2,2,2,5,5,5,8]`, `k = 3`, `threshold = 4` ⇒ cần **tổng ≥ 12**.

| i | cửa sổ | tổng | ≥ 12? |
|---|---|---|---|
| 0 | [2,2,2] | 6 | ✗ |
| 1 | [2,2,2] | 6 | ✗ |
| 2 | [2,2,5] | 9 | ✗ |
| 3 | [2,5,5] | 12 | ✔ |
| 4 | [5,5,5] | 15 | ✔ |
| 5 | [5,5,8] | 18 | ✔ |

Đếm được **3** ✔. Quan sát: hai cửa sổ liên tiếp **chung `k − 1` phần tử** ⇒ tổng mới = tổng cũ + phần tử mới − phần tử cũ.

In [7]:
arr, k, t, want = examples[0]
target = k * t
for i in range(len(arr) - k + 1):
    w = arr[i:i + k]
    print(f"i={i}: {w} tổng={sum(w):2} {'≥' if sum(w) >= target else '<'} {target}")

i=0: [2, 2, 2] tổng= 6 < 12
i=1: [2, 2, 2] tổng= 6 < 12
i=2: [2, 2, 5] tổng= 9 < 12
i=3: [2, 5, 5] tổng=12 ≥ 12
i=4: [5, 5, 5] tổng=15 ≥ 12
i=5: [5, 5, 8] tổng=18 ≥ 12


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** với mỗi vị trí bắt đầu `i`, tính tổng `k` phần tử bằng vòng lặp (hoặc `sum(arr[i:i+k])`), so sánh với `k·threshold`.

In [8]:
def num_of_subarrays_brute(arr, k, threshold):
    n = len(arr)
    target = k * threshold
    count = 0
    for i in range(n - k + 1):
        s = 0
        for j in range(i, i + k):
            s += arr[j]
        if s >= target:
            count += 1
    return count

for arr, k, t, want in examples:
    print(num_of_subarrays_brute(arr, k, t), "| mong đợi", want)

3 | mong đợi 3
6 | mong đợi 6


**Độ phức tạp:** `(n − k + 1)` cửa sổ × `k` phép cộng ⇒ `O(n·k)`; xấu nhất `k ≈ n/2` ⇒ `O(n²/4)` ≈ `2.5·10^9` với `n = 10^5` — quá chậm. Bộ nhớ `O(1)`.

**Nút thắt:** mỗi cửa sổ cộng lại `k` phần tử, trong khi `k − 1` phần tử đã được cộng ở cửa sổ trước. Hai cách tái sử dụng: **prefix sum** (tính trước mọi tổng) hoặc **cửa sổ trượt** (cập nhật tăng dần).

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (số nguyên):** điều kiện `avg >= threshold` ⇔ `window_sum >= k * threshold`. Tính `target = k * threshold` **một lần**.

**Insight 2 (cửa sổ trượt):** gọi `S_i` = tổng cửa sổ bắt đầu ở `i`. Khi đó `S_{i+1} = S_i + arr[i+k] − arr[i]`.

**Bất biến:** ở đầu mỗi bước, `window` là tổng **đúng `k` phần tử liên tiếp** kết thúc ở chỉ số hiện tại.

**Quy tắc "nếu… thì…":**
- Khởi tạo `window = sum(arr[:k])`; nếu `window >= target` thì `count = 1`.
- Với `i = k..n−1`: `window += arr[i] − arr[i−k]`; nếu `window >= target` thì `count += 1`.

**Insight 3 (trừ ngưỡng):** đặt `b[i] = arr[i] − threshold`. Trung bình đoạn `≥ threshold` ⇔ tổng `b` trên đoạn `≥ 0`. Hữu ích khi độ dài không cố định (xem Bài tập 5).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tổng tiền tố

**Từng đoạn:**
1. `P = [0] + tổng tích luỹ` (độ dài `n + 1`).
2. Cửa sổ bắt đầu `i` có tổng `P[i+k] − P[i]`.
3. Đếm `i ∈ [0, n−k]` thoả `P[i+k] − P[i] >= k·threshold`.

In [9]:
def num_of_subarrays_prefix(arr, k, threshold):
    P = [0] * (len(arr) + 1)
    for i, x in enumerate(arr):
        P[i + 1] = P[i] + x
    target = k * threshold
    return sum(1 for i in range(len(arr) - k + 1) if P[i + k] - P[i] >= target)

for arr, k, t, want in examples:
    print(num_of_subarrays_prefix(arr, k, t), "| mong đợi", want)

3 | mong đợi 3
6 | mong đợi 6


Code nộp LeetCode (Cách 2):

In [10]:
class SolutionPrefix:
    def numOfSubarrays(self, arr: list[int], k: int, threshold: int) -> int:
        P = list(accumulate(arr, initial=0))
        target = k * threshold
        return sum(P[i + k] - P[i] >= target for i in range(len(arr) - k + 1))

print([SolutionPrefix().numOfSubarrays(*e[:3]) for e in examples])

[3, 6]


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ cho mảng `P`. Ưu điểm: trả lời được tổng của **bất kỳ** đoạn nào trong `O(1)` (hữu ích khi có nhiều `k` khác nhau).

### 7.2 Cách 3 — Cửa sổ trượt kích thước cố định (lời giải chuẩn)

**Từng đoạn:**
1. `target = k * threshold`, `window = sum(arr[:k])`, `count = int(window >= target)`.
2. Duyệt `i` từ `k` đến `n − 1`: `window += arr[i] − arr[i − k]`; nếu `window >= target` thì `count += 1`.
3. Trả `count`.

In [11]:
def num_of_subarrays_window(arr, k, threshold):
    target = k * threshold
    window = sum(arr[:k])
    count = 1 if window >= target else 0
    for i in range(k, len(arr)):
        window += arr[i] - arr[i - k]
        if window >= target:
            count += 1
    return count

for arr, k, t, want in examples:
    print(num_of_subarrays_window(arr, k, t), "| mong đợi", want)

3 | mong đợi 3
6 | mong đợi 6


Code nộp LeetCode (Cách 3):

In [12]:
class SolutionWindow:
    def numOfSubarrays(self, arr: list[int], k: int, threshold: int) -> int:
        target = k * threshold
        window = sum(arr[:k])
        count = int(window >= target)
        for i in range(k, len(arr)):
            window += arr[i] - arr[i - k]
            count += window >= target
        return count

print([SolutionWindow().numOfSubarrays(*e[:3]) for e in examples])

[3, 6]


**Độ phức tạp:** `O(n)` thời gian (mỗi phần tử được cộng một lần, trừ một lần), `O(1)` bộ nhớ phụ.

### 7.3 Cách 4 — Trừ ngưỡng rồi trượt cửa sổ "tổng ≥ 0"

**Ý tưởng:** thay `arr[i]` bằng `b[i] = arr[i] − threshold`. Khi đó cửa sổ hợp lệ ⇔ tổng `b` của cửa sổ `≥ 0`. Code gần như Cách 3, nhưng tư duy "trừ ngưỡng" mở rộng được cho bài "trung bình ≥ threshold với **độ dài bất kỳ**" (khi đó không còn nhân `k` được).

Viết theo kiểu một vòng lặp duy nhất: cộng phần tử mới, nếu cửa sổ đã đủ `k` phần tử thì kiểm tra rồi bỏ phần tử trái.

In [13]:
def num_of_subarrays_shift(arr, k, threshold):
    count = 0
    window = 0
    for i, x in enumerate(arr):
        window += x - threshold                 # b[i]
        if i >= k:
            window -= arr[i - k] - threshold    # bỏ b[i-k]
        if i >= k - 1 and window >= 0:
            count += 1
    return count

for arr, k, t, want in examples:
    print(num_of_subarrays_shift(arr, k, t), "| mong đợi", want)

3 | mong đợi 3
6 | mong đợi 6


Code nộp LeetCode (Cách 4):

In [14]:
class SolutionShift:
    def numOfSubarrays(self, arr: list[int], k: int, threshold: int) -> int:
        count = window = 0
        for i, x in enumerate(arr):
            window += x - threshold
            if i >= k:
                window -= arr[i - k] - threshold
            if i >= k - 1 and window >= 0:
                count += 1
        return count

print([SolutionShift().numOfSubarrays(*e[:3]) for e in examples])

[3, 6]


**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

## 8. TRACE từng bước

Trace Cách 3 trên **Ví dụ 2** (`k = 3`, `threshold = 5` ⇒ `target = 15`):

| i | thêm | bỏ | window | ≥ 15? | count |
|---|---|---|---|---|---|
| khởi tạo | 11,13,17 | — | 41 | ✔ | 1 |
| 3 | 23 | 11 | 53 | ✔ | 2 |
| 4 | 29 | 13 | 69 | ✔ | 3 |
| 5 | 31 | 17 | 83 | ✔ | 4 |
| 6 | 7 | 23 | 67 | ✔ | 5 |
| 7 | 5 | 29 | 43 | ✔ | 6 |
| 8 | 2 | 31 | 14 | ✗ | 6 |
| 9 | 3 | 7 | 10 | ✗ | 6 |

In [15]:
def trace_window(arr, k, threshold):
    target = k * threshold
    window = sum(arr[:k])
    count = int(window >= target)
    print(f"target = {k}*{threshold} = {target}")
    print(f"khởi tạo: cửa sổ {arr[:k]} window={window} -> count={count}")
    for i in range(k, len(arr)):
        window += arr[i] - arr[i - k]
        ok = window >= target
        count += ok
        print(f"i={i}: thêm {arr[i]:2}, bỏ {arr[i - k]:2} | cửa sổ {arr[i - k + 1:i + 1]} window={window:3} {'≥' if ok else '<'} {target} | count={count}")
    return count

trace_window(*examples[1][:3])

target = 3*5 = 15
khởi tạo: cửa sổ [11, 13, 17] window=41 -> count=1
i=3: thêm 23, bỏ 11 | cửa sổ [13, 17, 23] window= 53 ≥ 15 | count=2
i=4: thêm 29, bỏ 13 | cửa sổ [17, 23, 29] window= 69 ≥ 15 | count=3
i=5: thêm 31, bỏ 17 | cửa sổ [23, 29, 31] window= 83 ≥ 15 | count=4
i=6: thêm  7, bỏ 23 | cửa sổ [29, 31, 7] window= 67 ≥ 15 | count=5
i=7: thêm  5, bỏ 29 | cửa sổ [31, 7, 5] window= 43 ≥ 15 | count=6
i=8: thêm  2, bỏ 31 | cửa sổ [7, 5, 2] window= 14 < 15 | count=6
i=9: thêm  3, bỏ  7 | cửa sổ [5, 2, 3] window= 10 < 15 | count=6


6

Trace Cách 2 — in mảng tiền tố và từng hiệu `P[i+k] − P[i]` trên Ví dụ 1:

In [16]:
def trace_prefix(arr, k, threshold):
    P = list(accumulate(arr, initial=0))
    print("P =", P)
    target = k * threshold
    cnt = 0
    for i in range(len(arr) - k + 1):
        s = P[i + k] - P[i]
        cnt += s >= target
        print(f"i={i}: P[{i + k}]-P[{i}] = {P[i + k]}-{P[i]} = {s:2} -> {'đếm' if s >= target else 'bỏ'} (count={cnt})")
    return cnt

trace_prefix(*examples[0][:3])

P = [0, 2, 4, 6, 8, 13, 18, 23, 31]
i=0: P[3]-P[0] = 6-0 =  6 -> bỏ (count=0)
i=1: P[4]-P[1] = 8-2 =  6 -> bỏ (count=0)
i=2: P[5]-P[2] = 13-4 =  9 -> bỏ (count=0)
i=3: P[6]-P[3] = 18-6 = 12 -> đếm (count=1)
i=4: P[7]-P[4] = 23-8 = 15 -> đếm (count=2)
i=5: P[8]-P[5] = 31-13 = 18 -> đếm (count=3)


3

## 9. Code nộp LeetCode

In [17]:
class Solution:
    def numOfSubarrays(self, arr: list[int], k: int, threshold: int) -> int:
        target = k * threshold                     # avg >= t  <=>  sum >= k*t
        window = sum(arr[:k])
        count = 1 if window >= target else 0
        for i in range(k, len(arr)):
            window += arr[i] - arr[i - k]          # thêm phải, bỏ trái
            if window >= target:
                count += 1
        return count

for arr, k, t, want in examples:
    got = Solution().numOfSubarrays(arr, k, t)
    print(got, "| mong đợi", want, "| OK" if got == want else "| SAI")

3 | mong đợi 3 | OK
6 | mong đợi 6 | OK


## 10. Vì sao thuật toán đúng

1. **Tương đương điều kiện:** với `k ≥ 1`, `S/k ≥ threshold ⇔ S ≥ k·threshold` (nhân hai vế với số dương). Mọi đại lượng là số nguyên ⇒ phép so sánh chính xác tuyệt đối.
2. **Bất biến cửa sổ:** trước vòng lặp, `window = S_0` (tổng `arr[0..k−1]`). Giả sử trước bước `i` (`i ≥ k`), `window = S_{i−k}` = tổng `arr[i−k..i−1]`. Sau `window += arr[i] − arr[i−k]` ta có tổng `arr[i−k+1..i] = S_{i−k+1}` ⇒ bất biến giữ (quy nạp).
3. **Đếm đủ, không trùng:** các giá trị `S_0, S_1, …, S_{n−k}` lần lượt được kiểm tra đúng một lần (một lần trước vòng lặp, `n − k` lần trong vòng lặp) ⇒ đúng `n − k + 1` cửa sổ.
4. **Prefix sum:** `P[i+k] − P[i] = Σ arr[i..i+k−1] = S_i` theo định nghĩa `P`.

## 11. Phân tích Time / Space

Đặt `n = len(arr)` (`≤ 10^5`).

| Cách | Time | Space phụ |
|---|---|---|
| 1. Brute force | `O(n·k)` (xấu nhất `O(n²)`) | `O(1)` |
| 2. Prefix sum | `O(n)` | `O(n)` |
| 3. Cửa sổ trượt | `O(n)` | `O(1)` |
| 4. Trừ ngưỡng + cửa sổ | `O(n)` | `O(1)` |

Giá trị lớn nhất: tổng cửa sổ `≤ 10^5 · 10^4 = 10^9`, `k·threshold ≤ 10^9` — vừa với số nguyên 32-bit có dấu; Python thì không lo tràn.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
solvers = {
    "brute": num_of_subarrays_brute,
    "prefix": num_of_subarrays_prefix,
    "window": num_of_subarrays_window,
    "shift": num_of_subarrays_shift,
    "submit": lambda *a: Solution().numOfSubarrays(*a),
    "submit_prefix": lambda *a: SolutionPrefix().numOfSubarrays(*a),
    "submit_window": lambda *a: SolutionWindow().numOfSubarrays(*a),
    "submit_shift": lambda *a: SolutionShift().numOfSubarrays(*a),
}
cases = examples + [
    ([5], 1, 5, 1),                          # n = k = 1, đúng bằng ngưỡng
    ([5], 1, 6, 0),
    ([1, 1, 1, 1, 1], 1, 0, 5),              # threshold = 0 -> mọi cửa sổ
    ([1, 2, 3, 4], 4, 2, 1),                 # k = n: trung bình 2.5
    ([1, 2, 3, 4], 4, 3, 0),
    ([7, 7, 7, 7, 7, 7, 7], 7, 7, 1),
    ([4, 4, 4, 4], 2, 4, 3),                 # mọi cửa sổ đúng bằng ngưỡng
    ([10**4] * 6, 3, 10**4, 4),
]
for arr, k, t, want in cases:
    for name, f in solvers.items():
        assert f(arr, k, t) == want, (name, arr, k, t, f(arr, k, t), want)
print("Tất cả", len(cases), "case qua cho", len(solvers), "lời giải")

Tất cả 10 case qua cho 8 lời giải


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

Ngoài các cách trên, ta so thêm với một "trọng tài" dùng **phân số chính xác** (`fractions.Fraction`) để chắc chắn việc đổi sang `sum >= k·threshold` không sai.

In [19]:
from fractions import Fraction

def referee_fraction(arr, k, threshold):
    return sum(Fraction(sum(arr[i:i + k]), k) >= threshold for i in range(len(arr) - k + 1))

rng = random.Random(1343)
for _ in range(1500):
    n = rng.randint(1, 30)
    arr = [rng.randint(1, rng.choice([5, 20, 10**4])) for _ in range(n)]
    k = rng.randint(1, n)
    t = rng.randint(0, max(arr) + 1)
    want = referee_fraction(arr, k, t)
    for name, f in solvers.items():
        assert f(arr, k, t) == want, (name, arr, k, t)
print("1500 bộ ngẫu nhiên: mọi cách khớp trọng tài phân số chính xác")

1500 bộ ngẫu nhiên: mọi cách khớp trọng tài phân số chính xác


### 12.3 Kích thước lớn nhất `n = 10^5`

In [20]:
rng = random.Random(99)
big = [rng.randint(1, 10**4) for _ in range(10**5)]
for k, t in [(1, 5000), (50000, 5000), (10**5, 5001)]:
    res = {}
    for name in ["prefix", "window", "shift", "submit"]:
        t0 = time.time()
        res[name] = solvers[name](big, k, t)
        res[name + "_t"] = (time.time() - t0) * 1000
    assert len({res[x] for x in ["prefix", "window", "shift", "submit"]}) == 1
    print(f"k={k:6}, threshold={t}: đáp án {res['submit']:6} | " + ", ".join(f"{x} {res[x + '_t']:.0f}ms" for x in ["prefix", "window", "shift", "submit"]))

k=     1, threshold=5000: đáp án  49813 | prefix 17ms, window 9ms, shift 14ms, submit 9ms


k= 50000, threshold=5000: đáp án   2127 | prefix 13ms, window 7ms, shift 18ms, submit 7ms
k=100000, threshold=5001: đáp án      0 | prefix 10ms, window 1ms, shift 8ms, submit 1ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `>` thay vì `>=`

In [21]:
def wrong_strict(arr, k, threshold):
    target = k * threshold
    window = sum(arr[:k])
    count = int(window > target)
    for i in range(k, len(arr)):
        window += arr[i] - arr[i - k]
        count += window > target              # SAI
    return count

arr, k, t, want = examples[0]
print("sai:", wrong_strict(arr, k, t), "| đúng:", Solution().numOfSubarrays(arr, k, t))

sai: 2 | đúng: 3


**Vì sao sai:** cửa sổ `[2,5,5]` có trung bình **đúng bằng** 4 — đề yêu cầu "greater than **or equal**". **Sửa:** `>=`.

### Lỗi 2 — Bỏ sót cửa sổ cuối (`range(n − k)`)

In [22]:
def wrong_range(arr, k, threshold):
    target = k * threshold
    return sum(sum(arr[i:i + k]) >= target for i in range(len(arr) - k))     # SAI: thiếu +1

for arr, k, t, want in examples:
    print("sai:", wrong_range(arr, k, t), "| đúng:", Solution().numOfSubarrays(arr, k, t))

sai: 2 | đúng: 3
sai: 6 | đúng: 6


**Vì sao sai:** có `n − k + 1` cửa sổ (chỉ số bắt đầu `0..n−k`). `range(n − k)` dừng ở `n − k − 1` ⇒ mất cửa sổ cuối `[5,5,8]` ở Ví dụ 1. Với `k = n`, code sai còn trả `0` cửa sổ. **Sửa:** `range(len(arr) - k + 1)`.

### Lỗi 3 — Trượt cửa sổ nhưng bỏ nhầm phần tử

In [23]:
def wrong_slide(arr, k, threshold):
    target = k * threshold
    window = sum(arr[:k])
    count = int(window >= target)
    for i in range(k, len(arr)):
        window += arr[i] - arr[i - k + 1]     # SAI: phải bỏ arr[i - k]
        count += window >= target
    return count

for arr, k, t, want in examples:
    print("sai:", wrong_slide(arr, k, t), "| đúng:", Solution().numOfSubarrays(arr, k, t))

sai: 3 | đúng: 3
sai: 8 | đúng: 6


**Vì sao sai:** cửa sổ cũ là `arr[i−k..i−1]`; phần tử rơi ra khi thêm `arr[i]` là `arr[i−k]`. Bỏ `arr[i−k+1]` làm `window` không còn là tổng `k` phần tử liên tiếp (lệch dần sau mỗi bước). **Sửa:** `window += arr[i] - arr[i - k]`. Mẹo kiểm tra: thêm `assert window == sum(arr[i-k+1:i+1])` khi debug.

### Lỗi 4 — Tính lại `sum(arr[i:i+k])` mỗi bước (quá chậm)

In [24]:
def slow_slices(arr, k, threshold):
    target = k * threshold
    return sum(sum(arr[i:i + k]) >= target for i in range(len(arr) - k + 1))

arr = [rng.randint(1, 10**4) for _ in range(20000)]
k = 10000
t0 = time.time(); r1 = slow_slices(arr, k, 5000); ts = time.time() - t0
t0 = time.time(); r2 = Solution().numOfSubarrays(arr, k, 5000); tf = time.time() - t0
print(f"cùng kết quả {r1 == r2} | cắt lát: {ts * 1000:.0f} ms | cửa sổ trượt: {tf * 1000:.1f} ms (n chỉ 2·10^4)")

cùng kết quả True | cắt lát: 1135 ms | cửa sổ trượt: 1.6 ms (n chỉ 2·10^4)


**Vì sao sai:** mỗi lát cắt tốn `O(k)` (tạo list mới + cộng) ⇒ `O(n·k)`; với `n = 10^5`, `k = 5·10^4` là ~`2.5·10^9` thao tác — TLE dù kết quả đúng. **Sửa:** cửa sổ trượt hoặc prefix sum.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Mảng con **liên tiếp** độ dài **cố định `k`**" ⇒ **cửa sổ trượt cố định**: cộng phải, trừ trái.
- Điều kiện trên **trung bình** ⇒ nhân chéo thành điều kiện trên **tổng** (`sum >= k·t`); độ dài thay đổi ⇒ **trừ ngưỡng** rồi xét tổng `≥ 0`.
- Cần tổng của nhiều đoạn tuỳ ý ⇒ **prefix sum**.
- Luôn đếm số cửa sổ: `n − k + 1`; thử `k = 1` và `k = n`.
- Python: `count += cond` (bool cộng như 0/1) gọn và nhanh.

## 15. Biến thể của bài

1. **LeetCode 643 — Maximum Average Subarray I:** trung bình **lớn nhất** của cửa sổ độ dài `k` ⇒ cùng cửa sổ trượt, lấy `max(window)/k`.
2. **LeetCode 1456 — Maximum Number of Vowels in a Substring of Given Length:** cửa sổ cố định trên chuỗi.
3. **LeetCode 2090 — K Radius Subarray Averages:** trung bình mọi cửa sổ độ dài `2k+1`.
4. **Độ dài bất kỳ** (Bài tập 5): đếm mọi mảng con có trung bình `≥ threshold` ⇒ trừ ngưỡng + đếm cặp tiền tố `P[i] ≤ P[j]` (`O(n log n)`).
5. **LeetCode 644 — Maximum Average Subarray II** (Premium, độ dài `≥ k`): tìm nhị phân trên đáp án + trừ ngưỡng.

In [25]:
def find_max_average(nums, k):
    """LeetCode 643."""
    window = best = sum(nums[:k])
    for i in range(k, len(nums)):
        window += nums[i] - nums[i - k]
        best = max(best, window)
    return best / k

def k_radius_averages(nums, k):
    """LeetCode 2090: trung bình (chia nguyên) của cửa sổ tâm i bán kính k, -1 nếu không đủ."""
    n, w = len(nums), 2 * k + 1
    res = [-1] * n
    if w > n:
        return res
    window = sum(nums[:w])
    res[k] = window // w
    for i in range(w, n):
        window += nums[i] - nums[i - w]
        res[i - k] = window // w
    return res

print(find_max_average([1, 12, -5, -6, 50, 3], 4), k_radius_averages([7, 4, 3, 9, 1, 8, 5, 2, 6], 3))

12.75 [-1, -1, -1, 5, 4, 4, -1, -1, -1]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | cộng lại `k` phần tử cho mỗi cửa sổ | `O(n·k)` | `O(1)` | kiểm chứng, `n` nhỏ |
| 2. Prefix sum | `P[i+k] − P[i]` | `O(n)` | `O(n)` | nhiều truy vấn đoạn / nhiều `k` |
| 3. Cửa sổ trượt | cộng phải, trừ trái | `O(n)` | `O(1)` | lời giải chuẩn |
| 4. Trừ ngưỡng | `b = arr − t`, tổng `≥ 0` | `O(n)` | `O(1)` | chuẩn bị cho độ dài bất kỳ |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`numOfSubarrays([1,1,1,1,1], k=1, threshold=0)` và `numOfSubarrays([3,3,3], k=2, threshold=4)` trả về gì?

### Bài 2 (tính tay)
`arr = [8,1,6,2,9,3], k = 2, threshold = 5`. Lập bảng cửa sổ trượt (window, so sánh với `target`) và đếm.

### Bài 3 (điền TODO)
Hoàn thiện `window_todo` theo Cách 3.

### Bài 4 (tự viết lại)
Viết lại Cách 2 (prefix sum) **không dùng `accumulate`**, đặt tên `prefix_ex4`.

### Bài 5 (mini challenge)
Viết `count_any_length(arr, threshold)` đếm **mọi** mảng con liên tiếp (độ dài bất kỳ `≥ 1`) có trung bình `≥ threshold` trong `O(n log n)`. Gợi ý: trừ ngưỡng, rồi đếm cặp `i < j` với `P[i] ≤ P[j]`. Kiểm chứng bằng brute force `O(n²)`.

In [26]:
# Bài 3, 4, 5 — khung (chỉ định nghĩa, không gọi)
def window_todo(arr, k, threshold):
    target = None          # TODO
    window = None          # TODO: tổng k phần tử đầu
    count = 0
    # TODO: trượt cửa sổ và đếm
    return count

def prefix_ex4_todo(arr, k, threshold):
    # TODO
    pass

def count_any_length_todo(arr, threshold):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) `threshold = 0` ⇒ mọi cửa sổ hợp lệ. (2) `k = 1` ⇒ 5 cửa sổ. (3) `[3,3,3]`, `k = 2`: mọi tổng là 6 < 8 ⇒ 0.

**Bài 2:** (1) `target = 10`. (2) Các tổng: `9, 7, 8, 11, 12`. (3) Đếm `≥ 10` ⇒ 2.

**Bài 3:** (1) `target = k * threshold`. (2) `window = sum(arr[:k])`, `count = int(window >= target)`. (3) `for i in range(k, n): window += arr[i] - arr[i-k]; count += window >= target`.

**Bài 4:** (1) `P = [0] * (n + 1)`. (2) `P[i+1] = P[i] + arr[i]`. (3) Đếm `P[i+k] - P[i] >= k*threshold`.

**Bài 5:** (1) `b[i] = arr[i] − threshold`, `P` là tiền tố của `b` (có `P[0] = 0`). (2) Mảng con `(i, j]` hợp lệ ⇔ `P[j] − P[i] ≥ 0` ⇔ `P[i] ≤ P[j]`. (3) Duyệt `j` tăng dần, giữ danh sách đã sắp các `P[i]` (`i < j`), đếm bằng `bisect_right`; hoặc dùng merge sort / Fenwick để đạt `O(n log n)` thật sự.

## 19. Đáp án tham khảo

In [27]:
# Bài 1, 2
b1 = (Solution().numOfSubarrays([1, 1, 1, 1, 1], 1, 0), Solution().numOfSubarrays([3, 3, 3], 2, 4))
print("Bài 1:", b1)
assert b1 == (5, 0)
print("Bài 2:")
assert trace_window([8, 1, 6, 2, 9, 3], 2, 5) == 2

# Bài 3
def window_sol(arr, k, threshold):
    target = k * threshold
    window = sum(arr[:k])
    count = int(window >= target)
    for i in range(k, len(arr)):
        window += arr[i] - arr[i - k]
        count += window >= target
    return count

# Bài 4
def prefix_ex4(arr, k, threshold):
    n = len(arr)
    P = [0] * (n + 1)
    for i in range(n):
        P[i + 1] = P[i] + arr[i]
    return sum(1 for i in range(n - k + 1) if P[i + k] - P[i] >= k * threshold)

for arr, k, t, want in cases:
    assert window_sol(arr, k, t) == want and prefix_ex4(arr, k, t) == want
print("Bài 1–4: qua")

Bài 1: (5, 0)
Bài 2:
target = 2*5 = 10
khởi tạo: cửa sổ [8, 1] window=9 -> count=0
i=2: thêm  6, bỏ  8 | cửa sổ [1, 6] window=  7 < 10 | count=0
i=3: thêm  2, bỏ  1 | cửa sổ [6, 2] window=  8 < 10 | count=0
i=4: thêm  9, bỏ  6 | cửa sổ [2, 9] window= 11 ≥ 10 | count=1
i=5: thêm  3, bỏ  2 | cửa sổ [9, 3] window= 12 ≥ 10 | count=2
Bài 1–4: qua


In [28]:
# Bài 5
def count_any_length(arr, threshold):
    """Đếm cặp i < j với P[i] <= P[j] bằng merge sort: O(n log n)."""
    P = [0]
    for x in arr:
        P.append(P[-1] + x - threshold)

    def sort_count(a):
        if len(a) <= 1:
            return a, 0
        mid = len(a) // 2
        left, cl = sort_count(a[:mid])
        right, cr = sort_count(a[mid:])
        cnt, j = cl + cr, 0
        for x in right:                               # đếm left[i] <= x
            while j < len(left) and left[j] <= x:
                j += 1
            cnt += j
        merged, i, j = [], 0, 0
        while i < len(left) and j < len(right):
            if left[i] <= right[j]:
                merged.append(left[i]); i += 1
            else:
                merged.append(right[j]); j += 1
        merged += left[i:] + right[j:]
        return merged, cnt

    return sort_count(P)[1]

def count_any_length_bisect(arr, threshold):
    """Cách đơn giản hơn với bisect.insort (O(n^2) xấu nhất do chèn list, nhưng ngắn)."""
    seen, cur, cnt = [0], 0, 0
    for x in arr:
        cur += x - threshold
        cnt += bisect.bisect_right(seen, cur)
        bisect.insort(seen, cur)
    return cnt

def count_any_length_brute(arr, threshold):
    n = len(arr)
    return sum(sum(arr[i:j]) >= threshold * (j - i) for i in range(n) for j in range(i + 1, n + 1))

print("Bài 5:", count_any_length([2, 2, 2, 2, 5, 5, 5, 8], 4), count_any_length_brute([2, 2, 2, 2, 5, 5, 5, 8], 4))
rng = random.Random(5)
for _ in range(500):
    arr = [rng.randint(1, 10) for _ in range(rng.randint(1, 25))]
    t = rng.randint(0, 11)
    want = count_any_length_brute(arr, t)
    assert count_any_length(arr, t) == want == count_any_length_bisect(arr, t)
print("Bài 5: 500 bộ ngẫu nhiên khớp brute force")

Bài 5: 15 15
Bài 5: 500 bộ ngẫu nhiên khớp brute force


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template cửa sổ trượt kích thước cố định:**

```python
target = k * threshold              # đổi trung bình -> tổng (số nguyên)
window = sum(arr[:k])
count = int(window >= target)
for i in range(k, len(arr)):
    window += arr[i] - arr[i - k]   # thêm phải, bỏ trái
    count += window >= target
return count
```

**Template prefix sum:** `P = list(accumulate(arr, initial=0))`; tổng `arr[i:i+k] = P[i+k] - P[i]`.

**Checklist trước khi submit:**
- [ ] So sánh `>=` (bao gồm bằng).
- [ ] Không dùng phép chia / số thực: `sum >= k * threshold`.
- [ ] Đếm đủ `n − k + 1` cửa sổ (thử `k = n`, `k = 1`).
- [ ] Trượt: bỏ đúng `arr[i - k]`.
- [ ] Không cắt lát `sum(arr[i:i+k])` trong vòng lặp.
- [ ] Đúng chữ ký: `numOfSubarrays(self, arr: list[int], k: int, threshold: int) -> int`.